# Fixed-transcript reliability sweep with Qwen3.5-4B

This notebook isolates the effect of the reliability stated to Qwen. It constructs a bank of fixed question/answer histories and replays every history under each reliability $r\in\{0.9,0.7,0.5,0.3,0.1\}$. The question and SOURCE-answer turns are byte-identical across the reliability sweep; only the reliability statement, exact Bayesian posterior, and normative target change.

**This notebook was authored but not executed.**

## Why use a fixed transcript bank?

If answers are sampled separately at every reliability, adaptive policies can follow different branches and ask different later questions. A measured difference can then reflect changed evidence or question difficulty rather than Qwen's use of $r$. Here, all $2^T$ YES/NO patterns are enumerated for each fixed question schedule. For adaptive policies, each answer pattern deterministically defines one complete branch, and that same branch is replayed at every $r$.

Two estimands are kept separate:

1. **Controlled macro accuracy:** every fixed transcript receives equal weight. This is the primary reliability-sensitivity test.
2. **Natural-distribution expected accuracy:** the same exhaustive transcripts are weighted by their exact prior-predictive probability under each $r$. This estimates natural channel performance without resampling a different dataset at each reliability.

The second view is descriptive and must not replace the controlled comparison.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import random
from dataclasses import asdict, dataclass
from fractions import Fraction
from itertools import permutations, product
from pathlib import Path
from typing import Literal, Sequence

import matplotlib.pyplot as plt

PolicyName = Literal["binary_search", "random_memoryless", "random_elimination"]
Answer = Literal["YES", "NO"]
Label = Literal["A", "B", "C"]
SemanticChoice = Literal["left", "right", "tie"]
LABELS: tuple[Label, ...] = ("A", "B", "C")
SEMANTIC_CHOICES: tuple[SemanticChoice, ...] = ("left", "right", "tie")
LABEL_ASSIGNMENTS = tuple(permutations(SEMANTIC_CHOICES))


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the MATS repository.")


REPO_ROOT = find_repo_root()
MODEL_PATH = REPO_ROOT / "models" / "Qwen--Qwen3.5-4B"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "fixed_transcript_reliability"

N = 8
NUM_TURNS = 3
RELIABILITIES = tuple(Fraction(x, 10) for x in (9, 7, 5, 3, 1))
POLICIES: tuple[PolicyName, ...] = (
    "binary_search",
    "random_memoryless",
    "random_elimination",
)
CANDIDATE_PAIR = (2, 7)
NUM_RANDOM_QUESTION_SCHEDULES = 1
BASE_SEED = 20260826

assert N >= 2 and NUM_TURNS >= 1
assert all(Fraction(0) < r < Fraction(1) for r in RELIABILITIES)
assert len(set(CANDIDATE_PAIR)) == 2 and all(1 <= x <= N for x in CANDIDATE_PAIR)

## Fixed bank and replay types

A bank item has no generating secret or generating reliability. It is an exogenously designed observation history that has nonzero probability under every tested channel. The hidden secret is marginalized under the uniform prior when its posterior and prior-predictive probability are computed.

In [ ]:
@dataclass(frozen=True)
class Observation:
    turn: int
    subset: tuple[int, ...]
    answer: Answer


@dataclass(frozen=True)
class TranscriptBankItem:
    bank_id: str
    policy: PolicyName
    schedule_id: int
    answer_pattern: tuple[Answer, ...]
    observations: tuple[Observation, ...]
    turn_messages: tuple[dict[str, str], ...]


@dataclass(frozen=True)
class Probe:
    kind: Literal["candidate", "half"]
    question: str
    option_texts: tuple[str, str, str]
    normative_semantic_choice: SemanticChoice
    option_probabilities: tuple[Fraction, Fraction]


@dataclass(frozen=True)
class ReplayExample:
    replay_id: str
    bank_id: str
    policy: PolicyName
    schedule_id: int
    reliability: Fraction
    posterior: tuple[Fraction, ...]
    prior_predictive_probability: Fraction
    messages: tuple[dict[str, str], ...]
    probes: tuple[Probe, ...]


SYSTEM_TEMPLATE = """You are in a hidden-secret inference game. You are talking to SOURCE.
The secret s was drawn uniformly from the integers 1 through {n}.
SOURCE knows s. For each membership question, the proposition is either true or false.
SOURCE reports the proposition's true YES/NO answer with probability r = {reliability}, and reports the opposite answer with probability 1-r = {error_rate}.
SOURCE's channel outcomes are independent between questions conditional on s. The value of r may be below 0.5.
The questions were selected externally by a scripted policy. Conditional on the visible earlier answers, question selection carries no additional information about s.
Treat each message beginning 'SOURCE answer:' as an observation from the stated noisy channel.
At the final TEST, compare the two posterior probabilities requested. Reply with exactly A, B, or C and no other text."""


def format_set(values: Sequence[int]) -> str:
    return "{" + ", ".join(map(str, values)) + "}"


def system_message(n: int, reliability: Fraction) -> dict[str, str]:
    return {
        "role": "system",
        "content": SYSTEM_TEMPLATE.format(
            n=n,
            reliability=float(reliability),
            error_rate=float(1 - reliability),
        ),
    }


def question_message(turn: int, subset: Sequence[int]) -> dict[str, str]:
    return {
        "role": "assistant",
        "content": (
            f"Question {turn}: Is the secret in {format_set(subset)}? "
            "Answer YES or NO."
        ),
    }


def answer_message(answer: Answer) -> dict[str, str]:
    return {"role": "user", "content": f"SOURCE answer: {answer}."}

## Construct fixed question/answer histories

Question randomness is keyed only by policy and schedule ID—never by the secret or evaluated reliability. Binary search has one deterministic schedule. Random policies may use multiple independently fixed schedules. Every YES/NO pattern is enumerated, and adaptive policy state is updated from that fixed visible pattern.

In [ ]:
def stable_seed(base_seed: int, *parts: object) -> int:
    payload = "|".join(map(str, (base_seed, *parts))).encode()
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "big")


def choose_subset(
    policy: PolicyName,
    domain: tuple[int, ...],
    active: tuple[int, ...],
    rng: random.Random,
) -> tuple[tuple[int, ...], tuple[int, ...]]:
    scope = domain if policy == "random_memoryless" else (active if len(active) >= 2 else domain)
    subset_size = max(1, len(scope) // 2)
    if policy == "binary_search":
        subset = scope[:subset_size]
    else:
        subset = tuple(sorted(rng.sample(scope, k=subset_size)))
    assert 0 < len(subset) < len(scope)
    return subset, scope


def update_policy_state(
    policy: PolicyName,
    scope: tuple[int, ...],
    subset: tuple[int, ...],
    answer: Answer,
) -> tuple[int, ...]:
    if policy == "random_memoryless":
        return scope
    selected = set(subset)
    survivors = selected if answer == "YES" else set(scope) - selected
    return tuple(x for x in scope if x in survivors)


def build_bank_item(
    *, policy: PolicyName, schedule_id: int, answer_pattern: tuple[Answer, ...]
) -> TranscriptBankItem:
    domain = tuple(range(1, N + 1))
    active = domain
    rng = random.Random(stable_seed(BASE_SEED, "questions", policy, schedule_id))
    observations: list[Observation] = []
    turn_messages: list[dict[str, str]] = []
    for turn, answer in enumerate(answer_pattern, start=1):
        subset, scope = choose_subset(policy, domain, active, rng)
        observations.append(Observation(turn=turn, subset=subset, answer=answer))
        turn_messages.extend((question_message(turn, subset), answer_message(answer)))
        active = update_policy_state(policy, scope, subset, answer)

    answer_tag = "".join("Y" if answer == "YES" else "N" for answer in answer_pattern)
    bank_id = f"{policy}_schedule{schedule_id}_{answer_tag}"
    return TranscriptBankItem(
        bank_id=bank_id,
        policy=policy,
        schedule_id=schedule_id,
        answer_pattern=answer_pattern,
        observations=tuple(observations),
        turn_messages=tuple(turn_messages),
    )


ANSWER_PATTERNS: tuple[tuple[Answer, ...], ...] = tuple(
    product(("YES", "NO"), repeat=NUM_TURNS)
)
SCHEDULE_IDS: dict[PolicyName, tuple[int, ...]] = {
    "binary_search": (0,),
    "random_memoryless": tuple(range(NUM_RANDOM_QUESTION_SCHEDULES)),
    "random_elimination": tuple(range(NUM_RANDOM_QUESTION_SCHEDULES)),
}

transcript_bank = [
    build_bank_item(policy=policy, schedule_id=schedule_id, answer_pattern=answer_pattern)
    for policy in POLICIES
    for schedule_id in SCHEDULE_IDS[policy]
    for answer_pattern in ANSWER_PATTERNS
]
assert len({item.bank_id for item in transcript_bank}) == len(transcript_bank)
print(f"Built {len(transcript_bank)} fixed transcripts.")

## Replay each fixed history at every reliability

For candidate $x$, each answer contributes likelihood $r$ if it matches $[x\in Q_t]$ and $1-r$ otherwise. The normalization constant is also the prior-predictive probability of that fixed answer history under a uniform secret prior.

In [ ]:
def exact_posterior(
    n: int, observations: Sequence[Observation], reliability: Fraction
) -> tuple[tuple[Fraction, ...], Fraction]:
    weights: list[Fraction] = []
    for candidate in range(1, n + 1):
        weight = Fraction(1, n)
        for observation in observations:
            proposition = candidate in observation.subset
            reported_yes = observation.answer == "YES"
            weight *= reliability if proposition == reported_yes else 1 - reliability
        weights.append(weight)
    evidence_probability = sum(weights, start=Fraction(0))
    posterior = tuple(weight / evidence_probability for weight in weights)
    assert sum(posterior, start=Fraction(0)) == 1
    return posterior, evidence_probability


def compare(left: Fraction, right: Fraction) -> SemanticChoice:
    if left > right:
        return "left"
    if right > left:
        return "right"
    return "tie"


def make_probes(
    posterior: tuple[Fraction, ...], candidate_pair: tuple[int, int]
) -> tuple[Probe, Probe]:
    left_candidate, right_candidate = candidate_pair
    left_probability = posterior[left_candidate - 1]
    right_probability = posterior[right_candidate - 1]
    candidate_probe = Probe(
        kind="candidate",
        question="TEST — Given all evidence, which candidate has greater posterior probability?",
        option_texts=(
            str(left_candidate),
            str(right_candidate),
            "They have equal posterior probability.",
        ),
        normative_semantic_choice=compare(left_probability, right_probability),
        option_probabilities=(left_probability, right_probability),
    )

    midpoint = len(posterior) // 2
    first_probability = sum(posterior[:midpoint], start=Fraction(0))
    second_probability = sum(posterior[midpoint:], start=Fraction(0))
    half_probe = Probe(
        kind="half",
        question="TEST — Given all evidence, which set has greater posterior probability?",
        option_texts=(
            format_set(tuple(range(1, midpoint + 1))),
            format_set(tuple(range(midpoint + 1, len(posterior) + 1))),
            "The two sets have equal posterior probability.",
        ),
        normative_semantic_choice=compare(first_probability, second_probability),
        option_probabilities=(first_probability, second_probability),
    )
    return candidate_probe, half_probe


def render_probe(probe: Probe, semantics_by_label: Sequence[SemanticChoice]) -> str:
    if set(semantics_by_label) != set(SEMANTIC_CHOICES):
        raise ValueError("Each assignment must contain left, right, and tie once.")
    lines = [probe.question]
    for label, semantic in zip(LABELS, semantics_by_label):
        lines.append(f"{label}: {probe.option_texts[SEMANTIC_CHOICES.index(semantic)]}")
    lines.append("Reply with exactly A, B, or C.")
    return "\n".join(lines)


def replay_bank_item(item: TranscriptBankItem, reliability: Fraction) -> ReplayExample:
    posterior, evidence_probability = exact_posterior(N, item.observations, reliability)
    reliability_tag = f"{reliability.numerator}-{reliability.denominator}"
    return ReplayExample(
        replay_id=f"{item.bank_id}_r{reliability_tag}",
        bank_id=item.bank_id,
        policy=item.policy,
        schedule_id=item.schedule_id,
        reliability=reliability,
        posterior=posterior,
        prior_predictive_probability=evidence_probability,
        messages=(system_message(N, reliability), *item.turn_messages),
        probes=make_probes(posterior, CANDIDATE_PAIR),
    )


replays = [
    replay_bank_item(item, reliability)
    for item in transcript_bank
    for reliability in RELIABILITIES
]
print(f"Created {len(replays)} reliability replays and {2 * len(replays)} probes.")

## Design invariants

These assertions establish the key causal control: within a bank ID, all non-system turns and all probe option meanings are identical across $r$. They also verify that enumerated answer histories have total prior-predictive probability one for every policy schedule and reliability.

In [ ]:
for item in transcript_bank:
    variants = [replay for replay in replays if replay.bank_id == item.bank_id]
    assert len(variants) == len(RELIABILITIES)
    assert all(replay.messages[1:] == variants[0].messages[1:] for replay in variants)
    assert all(
        tuple(probe.option_texts for probe in replay.probes)
        == tuple(probe.option_texts for probe in variants[0].probes)
        for replay in variants
    )

for policy in POLICIES:
    for schedule_id in SCHEDULE_IDS[policy]:
        for reliability in RELIABILITIES:
            total_probability = sum(
                (replay.prior_predictive_probability for replay in replays
                 if replay.policy == policy
                 and replay.schedule_id == schedule_id
                 and replay.reliability == reliability),
                start=Fraction(0),
            )
            assert total_probability == 1

assert all(
    probability == Fraction(1, N)
    for replay in replays
    if replay.reliability == Fraction(1, 2)
    for probability in replay.posterior
)
print("All fixed-transcript and probability-mass invariants passed.")

In [ ]:
def fraction_string(value: Fraction) -> str:
    return f"{value.numerator}/{value.denominator}"


ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
bank_path = ARTIFACT_DIR / "transcript_bank.jsonl"
with bank_path.open("w", encoding="utf-8") as handle:
    for item in transcript_bank:
        handle.write(json.dumps(asdict(item)) + "\n")

replay_path = ARTIFACT_DIR / "replays.jsonl"
with replay_path.open("w", encoding="utf-8") as handle:
    for replay in replays:
        record = {
            "replay_id": replay.replay_id,
            "bank_id": replay.bank_id,
            "policy": replay.policy,
            "schedule_id": replay.schedule_id,
            "reliability": fraction_string(replay.reliability),
            "posterior_exact": [fraction_string(p) for p in replay.posterior],
            "prior_predictive_probability": fraction_string(
                replay.prior_predictive_probability
            ),
            "messages": list(replay.messages),
            "probes": [
                {
                    "kind": probe.kind,
                    "semantic_options": dict(zip(SEMANTIC_CHOICES, probe.option_texts)),
                    "normative_semantic_choice": probe.normative_semantic_choice,
                    "option_probabilities_exact": [
                        fraction_string(p) for p in probe.option_probabilities
                    ],
                }
                for probe in replay.probes
            ],
        }
        handle.write(json.dumps(record) + "\n")
print(f"Saved fixed bank to {bank_path} and replays to {replay_path}.")

## Load Qwen3.5-4B locally — run only when ready

The following cells use the local checkpoint, disable thinking through Qwen's chat template, and score only the next-token labels A/B/C.

In [ ]:
import torch
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

assert MODEL_PATH.exists(), f"Missing local checkpoint: {MODEL_PATH}"
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    MODEL_DTYPE = torch.float16
else:
    DEVICE = torch.device("cpu")
    MODEL_DTYPE = torch.bfloat16

processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_PATH, dtype=MODEL_DTYPE, local_files_only=True
)
model.to(DEVICE)
model.eval()
print(f"Loaded {MODEL_PATH.name} on {DEVICE} with dtype={MODEL_DTYPE}.")

## Label-counterbalanced semantic scoring

Every probe is evaluated under all six assignments of `left`, `right`, and `tie` to A/B/C. Relative label log scores are mapped back to meaning and averaged. Permutation consistency reports the fraction of assignments whose mapped decision agrees with the aggregate semantic decision.

In [ ]:
LABEL_TOKEN_IDS: dict[Label, int] = {}
for label in LABELS:
    token_ids = processor.tokenizer(label, add_special_tokens=False)["input_ids"]
    if len(token_ids) != 1:
        raise ValueError(f"Expected one token for label {label}; got {token_ids}.")
    LABEL_TOKEN_IDS[label] = token_ids[0]


def relative_log_scores(log_scores: dict[str, float]) -> dict[str, float]:
    maximum = max(log_scores.values())
    log_normalizer = maximum + math.log(
        sum(math.exp(score - maximum) for score in log_scores.values())
    )
    return {key: score - log_normalizer for key, score in log_scores.items()}


def score_labels(messages: Sequence[dict[str, str]]) -> dict[Label, float]:
    prompt = processor.apply_chat_template(
        list(messages),
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    with torch.inference_mode():
        outputs = model(
            input_ids=prompt["input_ids"].to(DEVICE),
            attention_mask=prompt["attention_mask"].to(DEVICE),
            use_cache=False,
            logits_to_keep=1,
        )
        log_probs = outputs.logits[0, -1].float().log_softmax(dim=-1)
    return {label: float(log_probs[token_id].cpu()) for label, token_id in LABEL_TOKEN_IDS.items()}


def score_probe(replay: ReplayExample, probe: Probe) -> dict[str, object]:
    records: list[dict[str, object]] = []
    semantic_values: dict[SemanticChoice, list[float]] = {
        semantic: [] for semantic in SEMANTIC_CHOICES
    }
    for assignment in LABEL_ASSIGNMENTS:
        prompt = render_probe(probe, assignment)
        messages = [*replay.messages, {"role": "user", "content": prompt}]
        label_scores = score_labels(messages)
        relative_scores = relative_log_scores(label_scores)
        mapping = dict(zip(LABELS, assignment))
        predicted_label: Label = max(label_scores, key=label_scores.get)  # type: ignore[arg-type]
        predicted_semantic = mapping[predicted_label]
        for label, semantic in mapping.items():
            semantic_values[semantic].append(relative_scores[label])
        records.append(
            {
                "semantic_by_label": mapping,
                "label_log_scores": label_scores,
                "predicted_label": predicted_label,
                "predicted_semantic_choice": predicted_semantic,
            }
        )

    aggregate_scores: dict[str, float] = {
        semantic: sum(values) / len(values) for semantic, values in semantic_values.items()
    }
    aggregate_relative = relative_log_scores(aggregate_scores)
    aggregate_probabilities = {key: math.exp(value) for key, value in aggregate_relative.items()}
    predicted: SemanticChoice = max(aggregate_scores, key=aggregate_scores.get)  # type: ignore[arg-type]
    consistency = sum(
        record["predicted_semantic_choice"] == predicted for record in records
    ) / len(records)
    return {
        "replay_id": replay.replay_id,
        "bank_id": replay.bank_id,
        "policy": replay.policy,
        "schedule_id": replay.schedule_id,
        "reliability": float(replay.reliability),
        "probe_kind": probe.kind,
        "normative_semantic_choice": probe.normative_semantic_choice,
        "predicted_semantic_choice": predicted,
        "counterbalanced_correct": predicted == probe.normative_semantic_choice,
        "permutation_consistency": consistency,
        "aggregate_semantic_log_scores": aggregate_scores,
        "counterbalanced_semantic_probabilities": aggregate_probabilities,
        "prior_predictive_probability": float(replay.prior_predictive_probability),
        "permutations": records,
    }

## Run the replay grid — run only when ready

With the default three turns and one schedule per policy, the bank contains $3\times2^3=24$ transcripts. Five reliability replays, two probes, and six label assignments require 1,440 next-token forward passes.

In [ ]:
results: list[dict[str, object]] = []
for replay_index, replay in enumerate(replays, start=1):
    for probe in replay.probes:
        result = score_probe(replay, probe)
        results.append(result)
        print(
            f"[{replay_index:>3}/{len(replays)}] {replay.replay_id} {probe.kind}: "
            f"predicted={result['predicted_semantic_choice']} "
            f"normative={result['normative_semantic_choice']} "
            f"correct={result['counterbalanced_correct']} "
            f"consistency={result['permutation_consistency']:.2f}"
        )

results_path = ARTIFACT_DIR / "qwen_counterbalanced_results.jsonl"
with results_path.open("w", encoding="utf-8") as handle:
    for result in results:
        handle.write(json.dumps(result) + "\n")
print(f"Saved {len(results)} decisions to {results_path}.")

## Primary result: controlled macro accuracy

Every bank transcript is weighted equally below. Because each bank ID is replayed at every reliability, differences across $r$ cannot be attributed to different question/answer histories. The lower row shows permutation consistency.

In [ ]:
def group_key(row: dict[str, object]) -> tuple[str, str, float]:
    return str(row["policy"]), str(row["probe_kind"]), float(row["reliability"])


def grouped_mean(rows: Sequence[dict[str, object]], metric: str) -> dict[tuple[str, str, float], float]:
    groups: dict[tuple[str, str, float], list[float]] = {}
    for row in rows:
        groups.setdefault(group_key(row), []).append(float(row[metric]))
    return {key: sum(values) / len(values) for key, values in groups.items()}


macro_accuracy = grouped_mean(results, "counterbalanced_correct")
macro_consistency = grouped_mean(results, "permutation_consistency")
reliability_values = sorted(float(r) for r in RELIABILITIES)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex="col", sharey="row")
for column, probe_kind in enumerate(("candidate", "half")):
    for policy in POLICIES:
        axes[0, column].plot(
            reliability_values,
            [macro_accuracy[(policy, probe_kind, r)] for r in reliability_values],
            marker="o",
            label=policy,
        )
        axes[1, column].plot(
            reliability_values,
            [macro_consistency[(policy, probe_kind, r)] for r in reliability_values],
            marker="o",
            label=policy,
        )
    axes[0, column].set_title(f"{probe_kind.capitalize()} probe")
    axes[0, column].axhline(1 / 3, color="gray", linestyle=":", linewidth=1)
    axes[1, column].set_xlabel("Stated reliability r")
    for axis in axes[:, column]:
        axis.axvline(0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
        axis.set_ylim(-0.05, 1.05)
        axis.grid(alpha=0.25)
axes[0, 0].set_ylabel("Controlled counterbalanced accuracy")
axes[1, 0].set_ylabel("Permutation consistency")
axes[0, 1].legend(loc="best")
fig.suptitle("Fixed-transcript sensitivity to stated source reliability")
fig.tight_layout()
plt.show()

## Secondary result: exact natural-distribution weighting

For each policy schedule and reliability, the enumerated transcript probabilities sum to one. Weighting model outcomes by these exact probabilities asks how the same model would perform when transcripts occur naturally under that channel. These curves may differ because transcript prevalence and difficulty legitimately change with $r$, so they are reported separately from the controlled macro result.

In [ ]:
def grouped_weighted_mean(
    rows: Sequence[dict[str, object]], metric: str
) -> dict[tuple[str, str, float], float]:
    weighted_sums: dict[tuple[str, str, float], float] = {}
    weight_sums: dict[tuple[str, str, float], float] = {}
    for row in rows:
        key = group_key(row)
        schedule_count = len(SCHEDULE_IDS[str(row["policy"])])  # type: ignore[index]
        weight = float(row["prior_predictive_probability"]) / schedule_count
        weighted_sums[key] = weighted_sums.get(key, 0.0) + weight * float(row[metric])
        weight_sums[key] = weight_sums.get(key, 0.0) + weight
    return {key: weighted_sums[key] / weight for key, weight in weight_sums.items()}


natural_accuracy = grouped_weighted_mean(results, "counterbalanced_correct")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for axis, probe_kind in zip(axes, ("candidate", "half")):
    for policy in POLICIES:
        axis.plot(
            reliability_values,
            [natural_accuracy[(policy, probe_kind, r)] for r in reliability_values],
            marker="o",
            label=policy,
        )
    axis.axhline(1 / 3, color="gray", linestyle=":", linewidth=1)
    axis.axvline(0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
    axis.set_title(f"{probe_kind.capitalize()} probe")
    axis.set_xlabel("Stated reliability r")
    axis.set_ylim(-0.05, 1.05)
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Natural-weighted counterbalanced accuracy")
axes[1].legend(loc="best")
fig.suptitle("Expected performance under each natural channel distribution")
fig.tight_layout()
plt.show()

## Interpretation limits

Holding transcripts fixed isolates reliability sensitivity but deliberately includes histories that are rare under some channels. That is why controlled and natural-weighted results are separate. This design still tests forced-choice rankings rather than posterior calibration, and the fixed candidate/partition semantics should be expanded or world-label-counterbalanced before stronger claims.